In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
import torch.utils.data
import numpy as np
import os
import random
import clip  
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA 

In [ ]:
def sinkhorn_knopp(C, epsilon, iters=1000):
    """
    Sinkhorn-Knopp algorithm for optimal transport.
    Parameters:
        - C: Cost matrix (torch.Tensor) of shape (m,n). Since it is a simialirty matrix, we need to invert it C = -C (already changed the formulation directly for simplicity)
        - epsilon: entropy regularization parameter: epsilon = (1 / lambda). Higher values leads to more confident / sharp results (less evenly spread)
          e.g. a single point in the source distribution is mapped to a single point in the target distribution with an infinite amount of mass.
          These solutions are often impractical and do not provide meaningful insights. Smaller values lead to the opposite effect.
        - max_iters: Number of iterations

    Returns:
        - P: Optimal transport matrix of shape (m,n) (torch.Tensor)
    """
    # Initialize the transport matrix P
    P = torch.exp(C * epsilon)

    for _ in range(iters):
        # Row normalization
        P /= P.sum(dim=1, keepdim=True)

        # Column normalization
        P /= P.sum(dim=0, keepdim=True)

    return P

# 1. Extract Embeddings from the BioCLIP Model
# rna_embeds: [Batch, 512], protein_embeds: [Batch, 512]
# These vectors represent the biological features of RNA and Protein in a shared latent space.

# 2. Calculate the Raw Similarity Matrix
# This matrix serves as the 'Cost Matrix' (C) for the Sinkhorn algorithm.
# It measures the cosine similarity between every RNA and every Protein in the batch.
logits = rna_embeds @ protein_embeds.t() 

# 3. Apply Sinkhorn-Knopp Optimization
# This corresponds to the 'common_logits = sinkhorn_knopp(...)' logic in the original code.
# It refines the raw similarities into an "Optimal Transport" plan, ensuring 
# balanced matching across the entire batch (preventing 'hubness').
optimized_sim_matrix = sinkhorn_knopp(logits, epsilon=1/lambda_sinkhorn)

# 4. Final Prediction (Preparation for Validation)
# We extract the indices of the highest-scoring Protein for each RNA.
# These 'predicted_protein_indices' will be compared against the 'ground_truth_protein_indices'.
_, predicted_protein_indices = optimized_sim_matrix.topk(1, dim=-1)


In [ ]:
def get_intersection(X, Y):
    N = len(X)
    M = len(Y)
    
    # Calculate joint entropy H(X,Y)
    contingency_table = np.zeros((N, M))
    for i, x in enumerate(X):
        for j, y in enumerate(Y):
            if x == y:
                contingency_table[i, j] = 1
    
    contingency_table /= np.sum(contingency_table) 
    joint_entropy = -np.sum(np.sum(contingency_table * np.log(contingency_table + 1e-9)))
    mutual_info = np.log(N) + np.log(M) + joint_entropy
    
    if np.isnan(mutual_info):
        return 0
    
    return mutual_info


import torch
import numpy as np

def calculate_rna_protein_mi(predicted_indices, ground_truth_indices, num_classes):
    """
    Calculates the Mutual Information between predicted and actual proteins.
    
    predicted_indices: [Batch] - Protein indices predicted after Sinkhorn optimization.
    ground_truth_indices: [Batch] - The actual protein labels mapped to your RNA.
    num_classes: Total number of unique proteins in your dataset.
    """
    batch_size = predicted_indices.size(0)
    
    # 1. Create a Contingency Table (Joint Frequency)
    # This table counts how often 'Protein P' was predicted when 'Protein G' was the truth.
    contingency_table = torch.zeros(num_classes, num_classes).to(predicted_indices.device)
    for p, g in zip(predicted_indices, ground_truth_indices):
        contingency_table[p, g] += 1
    
    # 2. Convert to Joint Probability Distribution P(X, Y)
    p_xy = contingency_table / batch_size
    
    # 3. Calculate Joint Entropy H(X, Y)
    # We add 1e-9 to avoid log(0) errors.
    joint_entropy = -torch.sum(p_xy * torch.log(p_xy + 1e-9))
    
    # 4. Calculate Individual Entropies H(X) and H(Y)
    p_x = torch.sum(p_xy, dim=1) # Distribution of predictions
    p_y = torch.sum(p_xy, dim=0) # Distribution of ground truth
    
    ent_x = -torch.sum(p_x * torch.log(p_x + 1e-9))
    ent_y = -torch.sum(p_y * torch.log(p_y + 1e-9))
    
    # 5. Calculate Mutual Information: I(X; Y) = H(X) + H(Y) - H(X, Y)
    # This represents the information overlap between RNA and Protein.
    mi = ent_x + ent_y - joint_entropy
    
    return mi.item()

In [ ]:
def get_statistics(deletion_scores):
    auc = sum(deletion_scores) / len(deletion_scores)
    return auc

def get_deletion_score(concepts, all_descriptors, weight_dissection, class_id, topk_w):
    text_score_pairs = zip(concepts[0], concepts[1])
    text_score_pairs = sorted(text_score_pairs, key=lambda x: x[1], reverse=True)
    i_texts = [text for text, _ in text_score_pairs]
    i_indices = [all_descriptors.index(t) for t in i_texts]
    
    w_values, w_indices = weight_dissection[class_id].topk(topk_w)
    w_values, w_indices = w_values.tolist(), w_indices.tolist()

    sorted_indices_perturb = i_indices.copy()
    perturb_index = len(all_descriptors) + 1
    deletion_scores = []
    deletion_scores.append(get_intersection(sorted_indices_perturb, w_indices))

    for i in range(len(sorted_indices_perturb)):
        sorted_indices_perturb[i] = perturb_index   # smaller is better
        deletion_scores.append(get_intersection(sorted_indices_perturb, w_indices))
        
    return deletion_scores

def get_mean_scores(deletion_scores):
    # list containing the decreasing intersection scores for every image in the validation set
    max_len = max([len(l) for l in deletion_scores])  
    new_scores = []

    for l in deletion_scores:
        padding = max_len - len(l)
        padded_list = [0 for _ in range(padding)]
        new_scores.append(l + padded_list)

    new_scores = torch.FloatTensor(new_scores).mean(0)
    return new_scores.tolist()